In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.integrate import quad

**Ex. 1 a) Modelul probabilist.** Cele $n = 50$ de teste sunt independente conditionat de $\theta$, iar fiecare componenta este defecta cu probabilitatea $\theta$. Deci numarul de defecte este binomial:

$$Y \mid \theta \sim \mathrm{Binomial}(n, \theta), \qquad P(Y = y \mid \theta) = \binom{n}{y}\,\theta^{y}(1-\theta)^{n-y}$$

Functia de verosimilitate (in $\theta$, pentru $y = 8$ si $n = 50$) este $L(\theta) = P(Y = 8 \mid \theta) \propto \theta^{8}(1-\theta)^{42}$.

**b) Actualizarea Bayesiana.** Cu priorul $\theta \sim \mathrm{Beta}(2, 18)$, adica $p(\theta) \propto \theta^{1}(1-\theta)^{17}$, formula lui Bayes da

$$p(\theta \mid Y = y) \propto P(Y = y \mid \theta)\,p(\theta) \propto \theta^{y}(1-\theta)^{n-y} \cdot \theta^{\alpha-1}(1-\theta)^{\beta-1} = \theta^{\alpha + y - 1}(1-\theta)^{\beta + n - y - 1}$$

Aceasta este nucleul unei distributii Beta (familia Beta este conjugata pentru binomiala), deci

$$\theta \mid Y = y \sim \mathrm{Beta}(\alpha + y,\ \beta + n - y) = \mathrm{Beta}(2 + 8,\ 18 + 42) = \mathrm{Beta}(10, 60).$$

Constanta de normalizare se obtine din conditia ca densitatea sa integreze la 1, deci nu mai trebuie calculata explicit.

In [ ]:
# ex. 1 c): estimatori pentru theta
a0, b0 = 2, 18          # parametrii priorului beta
n, y = 50, 8            # 8 defecte din 50 de componente testate
a1, b1 = a0 + y, b0 + n - y   # parametrii posteriorului: beta(10, 60)

mle = y / n                              # maximul verosimilitatii binomiale
map_ = (a1 - 1) / (a1 + b1 - 2)          # modul posteriorului beta (a1, b1 > 1)
medie_post = a1 / (a1 + b1)              # media posteriorului
medie_prior = a0 / (a0 + b0)

print(f"posterior: Beta({a1}, {b1})")
print(f"MLE = {mle:.4f}")
print(f"MAP = {map_:.4f}")
print(f"media a posteriori = {medie_post:.4f}")
print(f"media a priori = {medie_prior:.4f}")

In [ ]:
# ex. 1 d): densitatile a priori si a posteriori pe acelasi grafic
theta = np.linspace(0, 0.5, 500)
plt.plot(theta, stats.beta.pdf(theta, a0, b0), label="a priori Beta(2, 18)")
plt.plot(theta, stats.beta.pdf(theta, a1, b1), label="a posteriori Beta(10, 60)")
plt.axvline(mle, color="gray", linestyle=":", label="MLE = 0.16")
plt.xlabel("theta (rata de defectare)")
plt.ylabel("densitate")
plt.legend()
plt.show()

# abaterile standard arata cat s-a micsorat incertitudinea
print("abatere standard a priori:", stats.beta.std(a0, b0))
print("abatere standard a posteriori:", stats.beta.std(a1, b1))

In [ ]:
# ex. 1 e): interval credibil central de 95%
inf, sup = stats.beta.ppf([0.025, 0.975], a1, b1)
print(f"interval credibil 95%: [{inf:.4f}, {sup:.4f}]")

# ex. 1 f): P(theta < 0.10 | Y = 8) = functia de repartitie a posteriorului in 0.10
p_sub_10 = stats.beta.cdf(0.10, a1, b1)
print(f"P(theta < 0.10 | Y = 8) = {p_sub_10:.4f}")
print(f"(a priori, P(theta < 0.10) = {stats.beta.cdf(0.10, a0, b0):.4f})")

**Ex. 1 g) Predictia Bayesiana.** Pentru $X \mid \theta \sim \mathrm{Binomial}(10, \theta)$ si $\theta \mid Y = 8 \sim \mathrm{Beta}(10, 60)$:

$$P(X = 0 \mid Y = 8) = \int_0^1 (1-\theta)^{10}\,\frac{\theta^{9}(1-\theta)^{59}}{B(10, 60)}\,d\theta = \frac{B(10, 70)}{B(10, 60)} = \prod_{k=0}^{9} \frac{60 + k}{70 + k}$$

Ultima egalitate vine din $B(a, b) = \Gamma(a)\Gamma(b)/\Gamma(a+b)$ si din $\Gamma(x + 10)/\Gamma(x) = \prod_{k=0}^{9}(x + k)$.

In [ ]:
# ex. 1 g): probabilitatea predictiva ca niciuna din cele 10 componente noi sa nu fie defecta
# formula inchisa: produs (b1 + k) / (a1 + b1 + k), k = 0..9
pred_formula = np.prod([(b1 + k) / (a1 + b1 + k) for k in range(10)])

# verificare prin integrare numerica a integralei din enunt
pred_integrala, _ = quad(lambda t: (1 - t) ** 10 * stats.beta.pdf(t, a1, b1), 0, 1)

print(f"formula inchisa: {pred_formula:.4f}")
print(f"integrare numerica: {pred_integrala:.4f}")

**Interpretare ex. 1.**

- **c)** MLE $= 0.160$, MAP $\approx 0.132$ si media a posteriori $\approx 0.143$. MLE foloseste doar datele (8/50), deci ignora priorul. MAP si media a posteriori combina datele cu priorul Beta(2, 18), care are media $0.10$, deci sunt "trase" spre $0.10$ fata de MLE. MAP este modul posteriorului, iar media este mai mare pentru ca distributia Beta(10, 60) este asimetrica spre dreapta.
- **d)** Datele au mutat densitatea spre valori mai mari (de la media $0.10$ la $0.143$) si au ingustat-o (abaterea standard scade de la $0.065$ la $0.042$): incertitudinea despre $\theta$ s-a redus.
- **e)** Intervalul credibil central de 95% este aproximativ $[0.072,\ 0.233]$: dat fiind modelul si datele, $\theta$ se afla in acest interval cu probabilitatea $0.95$. Spre deosebire de un interval de incredere clasic, aceasta este o afirmatie probabilista despre $\theta$.
- **f)** $P(\theta < 0.10 \mid Y = 8) \approx 0.148$ (fata de $0.58$ a priori). Datele nu sustin cerinta ca rata sa fie sub 10%: probabilitatea ca ea sa fie sub 10% este de doar aproximativ 15%.
- **g)** $P(X = 0 \mid Y = 8) \approx 0.236$. Este mai mare decat $(1 - 0.143)^{10} \approx 0.214$, obtinut cu o singura valoare a lui $\theta$ (media posteriorului). Predictia Bayesiana mediaza peste incertitudinea lui $\theta$, iar functia $(1-\theta)^{10}$ este convexa, deci media ei este mai mare decat valoarea ei in medie.

In [ ]:
# ex. 2 a): simularea unei ore
LAMBDA = 20 / 60      # sosiri pe minut (20 clienti pe ora)
MEDIE_PLATA = 2.0     # timpul mediu de plasare si plata, in minute
ORA = 60              # clientii care sosesc in [0, 60] sunt urmariti pana la capat

def genereaza(n_sim, seed, k_max=80):
    # genereaza n_sim ore de sosiri: timpii dintre sosiri sunt exponentiali (proces poisson)
    rng = np.random.default_rng(seed)
    A = np.cumsum(rng.exponential(1 / LAMBDA, (n_sim, k_max)), axis=1)
    # tinem doar sosirile din primele 60 de minute; k_max e suficient de mare ca sa nu taiem nimic
    assert (A[:, -1] > ORA).all()
    masca = A <= ORA
    k = masca.sum(axis=1).max()
    A, masca = A[:, :k], masca[:, :k]
    plata = rng.exponential(MEDIE_PLATA, (n_sim, k))   # timpii de plata
    gatit_unitar = rng.exponential(1.0, (n_sim, k))    # timpii de preparare cu media 1; ii inmultim cu alpha
    return A, masca, plata, gatit_unitar

def simuleaza(alpha, A, plata, gatit_unitar):
    # doua cozi in serie, ambele FIFO, cu cate un singur server:
    #   1) casa: clientul incepe sa plateasca cand ajunge SI casa e libera
    #   2) statia de gatit: comanda incepe sa fie preparata cand plata s-a terminat SI statia e libera
    # plata e FIFO, deci comenzile ajung la statia de gatit in aceeasi ordine ca sosirile,
    # iar la statie ordinea FIFO coincide cu ordinea clientilor
    n_sim, k = A.shape
    liber_casa = np.zeros(n_sim)     # momentul la care se elibereaza casa
    liber_gatit = np.zeros(n_sim)    # momentul la care se elibereaza statia de gatit
    start_plata = np.zeros_like(A)
    sfarsit_plata = np.zeros_like(A)
    start_gatit = np.zeros_like(A)
    primire = np.zeros_like(A)
    for j in range(k):   # clientii in ordinea sosirii, pentru toate simularile deodata
        start_plata[:, j] = np.maximum(A[:, j], liber_casa)
        sfarsit_plata[:, j] = start_plata[:, j] + plata[:, j]
        liber_casa = sfarsit_plata[:, j]
        start_gatit[:, j] = np.maximum(sfarsit_plata[:, j], liber_gatit)
        primire[:, j] = start_gatit[:, j] + alpha * gatit_unitar[:, j]
        liber_gatit = primire[:, j]
    return start_plata, sfarsit_plata, start_gatit, primire

# exemplu: o singura ora, cu alpha = 1 minut
A, masca, plata, gu = genereaza(1, seed=1)
sp, ep, sg, rec = simuleaza(1.0, A, plata, gu)
tabel = pd.DataFrame({
    "sosire": A[0], "start_plata": sp[0], "sfarsit_plata": ep[0],
    "start_gatit": sg[0], "primire": rec[0],
})[masca[0]].round(2)
tabel["timp_total"] = (tabel["primire"] - tabel["sosire"]).round(2)
print("clienti in ora simulata:", len(tabel))
tabel.head(8)

In [ ]:
# ex. 2 b): P(toti clientii primesc mancarea in cel mult 15 minute) in functie de alpha
N_SIM = 20000
SEED = 42
A, masca, plata, gu = genereaza(N_SIM, SEED)

def prob_toti(alpha, limita=15):
    # fractiunea de ore in care TOTI clientii sositi au timp de la sosire la primire <= limita
    _, _, _, rec = simuleaza(alpha, A, plata, gu)
    timp = np.where(masca, rec - A, 0)    # ignoram coloanele fara client
    return (timp.max(axis=1) <= limita).mean()

alphas = np.linspace(0.01, 3, 30)
plt.plot(alphas, [prob_toti(a) for a in alphas], label="limita 15 min")
plt.plot(alphas, [prob_toti(a, 25) for a in alphas], label="limita 25 min")
plt.axhline(0.95, color="red", linestyle="--", label="0.95")
plt.xlabel("alpha (minute)")
plt.ylabel("P(toti clientii <= limita)")
plt.legend()
plt.show()

print("P(toti <= 15 min) pentru alpha ~ 0:", prob_toti(1e-9))

def cauta_alpha(limita, prag=0.95, maxim=10.0):
    # cautare binara: cu aceleasi numere aleatoare, P scade cand alpha creste
    if prob_toti(1e-9, limita) < prag:
        return None      # nicio valoare a lui alpha nu atinge pragul
    lo, hi = 0.0, maxim
    for _ in range(30):
        mid = (lo + hi) / 2
        if prob_toti(mid, limita) >= prag:
            lo = mid
        else:
            hi = mid
    return lo

for limita in [15, 20, 25, 30]:
    print(f"limita {limita} min: alpha maxim =", cauta_alpha(limita))

In [ ]:
# ex. 2 c): timpul mediu de la sosire la primire si timpul mediu de asteptare (fara servire)
# lucram cu alpha din cazul limitei de 25 minute (pentru 15 minute nu exista solutie)
alpha_star = cauta_alpha(25)
sp, ep, sg, rec = simuleaza(alpha_star, A, plata, gu)

timp_total = (rec - A)[masca]
# asteptare = timpul din coada de la casa + timpul din coada la statia de gatit
asteptare = ((sp - A) + (sg - ep))[masca]

print(f"alpha = {alpha_star:.3f} minute")
print(f"timp mediu sosire -> primire: {timp_total.mean():.2f} minute")
print(f"timp mediu de asteptare (fara servire): {asteptare.mean():.2f} minute")
print(f"numar mediu de clienti pe ora: {masca.sum() / N_SIM:.2f}")

In [ ]:
# ex. 2 d): cat variaza alpha-ul gasit de la o rulare la alta (alte seminte, aceeasi metoda)
rezultate = []
for seed in range(1, 6):
    A, masca, plata, gu = genereaza(N_SIM, seed)
    rezultate.append(cauta_alpha(25))
print("alpha maxim pentru limita 25 min, cu 5 seminte diferite:")
print(np.round(rezultate, 3))
print("media:", np.mean(rezultate).round(3), "abaterea standard:", np.std(rezultate).round(3))

**Interpretare ex. 2.**

- **b)** Cu parametrii din enunt, **nu exista nicio valoare a lui $\alpha$** pentru care toti clientii dintr-o ora sa primeasca mancarea in cel mult 15 minute cu probabilitatea de cel putin 95%. Chiar si cu $\alpha \to 0$ (mancarea se prepara instantaneu), probabilitatea este de aproximativ $0.81$. Cauza este casa: cu $\lambda = 20$ clienti pe ora si plata de 2 minute, casa este ocupata $2/3$ din timp, iar in cursul unei ore unele cozi la casa depasesc singure 15 minute pentru cel putin un client. Cum probabilitatea scade cand $\alpha$ creste, $\alpha$ nu poate repara asta. Pentru alte limite, $\alpha$ maxim este: $20$ minute: nu exista, $25$ minute: $\approx 1.38$, $30$ minute: $\approx 1.98$.
- **c)** Pentru limita de 25 minute ($\alpha \approx 1.38$), timpul mediu de la sosire la primire este de aproximativ $7.3$ minute, din care aproximativ $3.9$ minute sunt asteptare in cozi, iar restul este servire (plata si prepararea).
- **d)** Am folosit $20\,000$ de simulari, semintele aleatoare fixe `42` (si `1..5` pentru verificare) si cautare binara pe $\alpha$, cu aceleasi numere aleatoare la fiecare pas, ca sa fie monotona functia $P(\alpha)$. Rezultatele difera de la o rulare la alta pentru ca simularea Monte Carlo are eroare aleatoare: cu $N$ simulari, o probabilitate $p \approx 0.95$ este estimata cu eroarea standard $\sqrt{p(1-p)/N} \approx 0.0015$, iar mici variatii ale lui $P$ se traduc in variatii ale lui $\alpha$. Cu 5 seminte diferite, $\alpha$ maxim variaza intre aproximativ 1.36 si 1.47. O seminta diferita, mai multe simulari sau o alta metoda de cautare (de exemplu o grila mai fina) dau valori usor diferite.